In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(6):
    if os.path.isdir(os.path.join(_root, "tools")) and os.path.isdir(os.path.join(_root, "data")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

from tools.paths import data_path, outputs_path

In [ ]:
# =====================================================================
# CULTURAL CORPS INTEGRATION - STEP 1 OF 5: INTAKE DIAGNOSTIC
# =====================================================================
# This is a standalone, read-only inspection script. It does not modify
# or depend on any other notebook cell - run it on its own, any time,
# without needing prior cells' variables in memory.
#
# WHY THIS EXISTS: Cultural Corps was cut from scope early in this
# project (it uses different hub-labeling than Career Launch/Spring
# Forward, so it never went through NIOCCS coding or the citywide
# comparison). This script is step 1 of bringing it back in - it just
# looks at the data and reports back, so we can decide how to clean and
# code it before touching anything else.
#
# OUTPUT: printed diagnostics only, plus one small reference CSV.
# Nothing here changes AllRoles_clean.csv, AllRoles_nioccs_ready.csv, or
# any file the rest of the pipeline depends on.
#
# NAMING CONVENTION for this workstream, to keep the data folder sane:
# every new file from here on is prefixed "cc_" and lives in its own
# subfolder, data/cultural_corps/ - so it's obvious at a glance
# which files belong to this later add-on effort versus the original
# pipeline.

import pandas as pd
import re
import os

IN_PATH = data_path("AllRoles_clean.csv")
OUT_DIR = data_path("cultural_corps")
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(IN_PATH, encoding="utf-8-sig", low_memory=False)
print(f"Total historical opportunities in AllRoles_clean.csv: {len(df)}")

# ------------------------------------------------------------------
# 1. What does "Opportunity Type Name" actually contain?
#    (Checking directly rather than assuming - this is the whole point
#    of doing this inspection first.)
# ------------------------------------------------------------------
print("\n" + "=" * 70)
print("Opportunity Type Name - full value counts")
print("=" * 70)
print(df["Opportunity Type Name"].value_counts(dropna=False).to_string())

# ------------------------------------------------------------------
# 2. Cross-check against Program Family, which we already derived from
#    Opportunity Campaign Name text parsing. Do these two fields agree,
#    or does Opportunity Type Name catch something different/additional?
# ------------------------------------------------------------------
print("\n" + "=" * 70)
print("Opportunity Type Name  x  Program Family (cross-tab)")
print("=" * 70)
print(pd.crosstab(df["Opportunity Type Name"], df["Program Family"]).to_string())

# Identify Cultural Corps rows both ways, so we can see if they match.
type_name_mask = df["Opportunity Type Name"].astype(str).str.contains("cultural corps", case=False, na=False)
program_family_mask = df["Program Family"] == "Cultural Corps"

n_type_name = type_name_mask.sum()
n_program_family = program_family_mask.sum()
n_agree = (type_name_mask & program_family_mask).sum()
n_type_only = (type_name_mask & ~program_family_mask).sum()
n_family_only = (~type_name_mask & program_family_mask).sum()

print(f"\nRows matching on 'Opportunity Type Name' contains 'Cultural Corps': {n_type_name}")
print(f"Rows matching on Program Family == 'Cultural Corps': {n_program_family}")
print(f"Agree (caught both ways): {n_agree}")
print(f"Caught by Type Name only: {n_type_only}")
print(f"Caught by Program Family only: {n_family_only}")
if n_type_only or n_family_only:
    print("NOTE: the two fields don't fully agree - worth deciding which one (or the union) "
          "is the right filter before Step 2.")

# Use the union of both signals for everything below, so nothing is missed
# at this inspection stage even if the two fields disagree.
cc_mask = type_name_mask | program_family_mask
cc = df[cc_mask].copy()
print(f"\nTotal Cultural Corps rows identified (union of both signals): {len(cc)}")

# ------------------------------------------------------------------
# 3. What do their "hub" labels actually look like? (We saw early on
#    that Cultural Corps uses season/batch labels like "Job Postings",
#    "January" instead of real sector names - confirming that here.)
# ------------------------------------------------------------------
print("\n" + "=" * 70)
print("Cultural Corps 'Hub Raw' values (the non-sector labels seen before)")
print("=" * 70)
print(cc["Hub Raw"].value_counts(dropna=False).head(20).to_string())

# ------------------------------------------------------------------
# 4. Coverage by year
# ------------------------------------------------------------------
print("\n" + "=" * 70)
print("Cultural Corps rows by Cohort Year")
print("=" * 70)
print(cc["Cohort Year"].value_counts(dropna=False).sort_index().to_string())

# ------------------------------------------------------------------
# 5. Title/description quality check - same questions asked of the 4
#    core hubs originally: blank, near-blank, or placeholder content?
# ------------------------------------------------------------------
cc["Description Word Count"] = cc["Opportunity Description"].apply(
    lambda t: len(str(t).split()) if pd.notna(t) else 0
)
n_blank_title = cc["Opportunity Name"].isna().sum() | (cc["Opportunity Name"].astype(str).str.strip() == "").sum()
n_blank_desc = (cc["Description Word Count"] == 0).sum()
n_near_blank_desc = ((cc["Description Word Count"] > 0) & (cc["Description Word Count"] < 15)).sum()

PLACEHOLDER_TITLE_RE = re.compile(
    r"enter your job title here|insert title of position here|organization name.*job position|insert name of org",
    re.I,
)
n_placeholder = cc["Opportunity Name"].apply(
    lambda t: bool(PLACEHOLDER_TITLE_RE.search(str(t))) if pd.notna(t) else False
).sum()

print("\n" + "=" * 70)
print("Title/description quality (same checks run on the 4 core hubs originally)")
print("=" * 70)
print(f"Blank titles: {n_blank_title}")
print(f"Blank descriptions: {n_blank_desc}")
print(f"Near-blank descriptions (1-14 words): {n_near_blank_desc}")
print(f"Placeholder/template titles: {n_placeholder}")
print(f"Average description word count: {cc['Description Word Count'].mean():.1f}")

# ------------------------------------------------------------------
# 6. Sample rows for manual eyeballing - title + description snippet
# ------------------------------------------------------------------
print("\n" + "=" * 70)
print("Sample Cultural Corps rows (for manual review)")
print("=" * 70)
sample = cc.sample(min(8, len(cc)), random_state=1)
for _, row in sample.iterrows():
    print(f"\nOpportunity Name: {row['Opportunity Name']!r}")
    print(f"Opportunity Type Name: {row['Opportunity Type Name']!r}")
    print(f"Hub Raw: {row['Hub Raw']!r}")
    print(f"Cohort Year: {row['Cohort Year']}")
    print(f"Description word count: {row['Description Word Count']}")
    print(f"Description (first 250 chars): {str(row['Opportunity Description'])[:250]!r}")

# ------------------------------------------------------------------
# Write one small reference file - not a pipeline input, just a saved
# copy of what this inspection found, for the next step to start from.
# ------------------------------------------------------------------
cc.to_csv(f"{OUT_DIR}/cc_step1_raw_cultural_corps_rows.csv", index=False)
print(f"\nWrote {OUT_DIR}/cc_step1_raw_cultural_corps_rows.csv ({len(cc)} rows) "
      f"- reference only, not yet cleaned or NIOCCS-ready.")

In [ ]:
# =====================================================================
# CULTURAL CORPS INTEGRATION - STEP 2 OF 5: CLEAN & PREPARE FOR NIOCCS
# =====================================================================
# Standalone, reads only from AllRoles_clean.csv - no other cell's
# variables required. Mirrors final_clean_for_nioccs.py's logic (same
# filters, same placeholder detection, same output shape) so Step 3
# (NIOCCS coding) can reuse the existing coding cell unmodified - just
# pointed at this file instead of AllRoles_nioccs_ready.csv.
#
# Per Step 1's findings:
#   - Filter on "Opportunity Type Name" == "Cultural Corps", NOT
#     Program Family - Type Name caught 29 real rows Program Family
#     missed, and caught everything Program Family did too.
#   - "Program Hub Category" (the i= industry parameter for NIOCCS) is
#     set uniformly to "Arts, Entertainment, and Recreation" for every
#     row - matching the original pattern of one fixed broad label per
#     program (same as "Healthcare", "STEM and Green", etc. originally),
#     and it's a real QCEW sector name unlike some of those.
#   - Descriptions here average 434 words (much longer than the core
#     hubs) and follow the same numbered Q&A intake format
#     (extract_duties_text in nioccs_api_call.py already handles this -
#     no new extraction logic needed. Worth raising MAX_DESCRIPTION_CHARS
#     in that script before running Step 3, given the length here.)

import pandas as pd
import re

IN_PATH = data_path("AllRoles_clean.csv")
OUT_PATH = data_path("cultural_corps", "cc_step2_ready_for_nioccs.csv")

HARDCODED_INDUSTRY = "Arts, Entertainment, and Recreation"

PLACEHOLDER_TITLE_RE = re.compile(
    r"enter your job title here|insert title of position here|"
    r"organization name.*job position|insert name of org",
    re.I,
)


def is_placeholder_title(title):
    if pd.isna(title):
        return False
    return bool(PLACEHOLDER_TITLE_RE.search(str(title)))


def main():
    df = pd.read_csv(IN_PATH, encoding="utf-8-sig", low_memory=False)
    n_start = len(df)
    print(f"Starting rows (all of AllRoles_clean.csv): {n_start}")

    # 1. Filter to Cultural Corps via Opportunity Type Name (per Step 1)
    df = df[df["Opportunity Type Name"] == "Cultural Corps"]
    print(f"After filtering to Opportunity Type Name == 'Cultural Corps': {len(df)}")
    if len(df) == 0:
        raise ValueError(
            "Zero rows matched Opportunity Type Name == 'Cultural Corps' - check the "
            "exact value in your data (Step 1's diagnostic found this exact string, but "
            "confirm it hasn't changed) before continuing."
        )
    n = len(df)

    # 2 & 3. Both title and description must be non-blank (same rule as
    # the core hubs)
    df = df[df["Opportunity Name"].notna() & (df["Opportunity Name"].str.strip() != "")]
    df = df[df["Opportunity Description"].notna() & (df["Opportunity Description"].str.strip() != "")]
    print(f"After requiring non-blank title AND description: {len(df)} (-{n - len(df)})")
    if len(df) == 0:
        raise ValueError("Zero rows remain after the non-blank title/description filter - stopping here "
                          "rather than continuing (pandas mishandles .apply() on an empty frame downstream).")
    n = len(df)

    # 4. Drop unfilled placeholder/template postings
    df = df[~df["Opportunity Name"].apply(is_placeholder_title)]
    print(f"After dropping placeholder/template titles: {len(df)} (-{n - len(df)})")
    if len(df) == 0:
        raise ValueError("Zero rows remain after the placeholder-title filter - stopping here.")
    n = len(df)

    # 5. Guarantee uniqueness on Opportunity Id
    df = df.drop_duplicates(subset=["Opportunity Id"], keep="first")
    print(f"After deduplicating on Opportunity Id: {len(df)} (-{n - len(df)})")

    print(f"\nFinal Cultural Corps dataset: {len(df)} rows "
          f"({len(df) / n_start * 100:.1f}% of the full historical file)")
    print("\nRows per cohort year:")
    print(df["Cohort Year"].value_counts(dropna=False).sort_index().to_string())

    # Hardcode the industry parameter, same pattern as the original 4 hubs.
    df["Program Hub Category"] = HARDCODED_INDUSTRY

    out = df[[
        "Opportunity Id", "Opportunity Name", "Opportunity Description",
        "Program Hub Category", "Cohort Year", "Program Family", "Agency Name",
    ]].rename(columns={
        "Opportunity Id": "Opportunity Identifier",
        "Opportunity Name": "Role Name",
        "Opportunity Description": "Role Description",
    })

    out.to_csv(OUT_PATH, index=False)
    print(f"\nWrote {OUT_PATH} ({len(out)} rows)")
    print("Ready for Step 3 - NIOCCS coding. Point the existing coding cell's "
          "READY_PATH / CODED_PATH at this file (and a new cc_step3_coded.csv "
          "output) rather than modifying AllRoles_nioccs_ready.csv directly.")

    return out


if __name__ == "__main__":
    main()

In [ ]:
# =====================================================================
# CULTURAL CORPS INTEGRATION - STEP 3 OF 5: NIOCCS CODING
# =====================================================================
# Fully standalone - duplicates the coding logic from nioccs_api_call.py
# rather than depending on that cell's functions being in memory. This
# is deliberate: the whole point of the cc_ series is to be able to come
# back to this later without needing the original notebook's state.
#
# Reads: cc_step2_ready_for_nioccs.csv (from Step 2)
# Writes: cc_step3_coded.csv
#
# Does NOT touch AllRoles_nioccs_ready.csv, AllRoles_nioccs_coded.csv, or
# any file the main pipeline depends on - this is a parallel file, merged
# in later (Step 4).
#
# NOTE ON MAX_DESCRIPTION_CHARS: raised to 3000 here (was 2000 in the
# original script) - Cultural Corps descriptions average 434 words
# (~2,500+ characters), noticeably longer than the core-hub postings
# this cap was originally tuned for.

import pandas as pd
import requests
import time
import json
import os
import re

READY_PATH = data_path("cultural_corps", "cc_step2_ready_for_nioccs.csv")
CHECKPOINT_PATH = data_path("cultural_corps", "cc_step3_coded.csv")

NIOCCS_BASE_URL = "https://wwwn.cdc.gov/nioccs/IOCode"
REQUEST_PARAMS_STATIC = {"c": 2, "v": 18, "u": 1, "n": 1}

MAX_DESCRIPTION_CHARS = 3000  # raised from 2000 - see note above

TEST_MODE = False
TEST_SAMPLE_SIZE = 10

SECONDS_BETWEEN_REQUESTS = 0.5
MAX_RETRIES_PER_ROW = 3

NAICS_INSUFFICIENT_INFO_CODE = "009990"
SOC_INSUFFICIENT_INFO_CODE = "00-9900"

# Same multi-anchor extraction as the main pipeline - Cultural Corps
# postings use the identical numbered Q&A intake format ("1. Please
# provide a description of this position:") we already solved for.
DUTIES_ANCHOR_PATTERNS = [
    re.compile(r"duties\s*&?\s*responsibilities\s*:?", re.I),
    re.compile(r"job responsibilities and tasks\s*:?", re.I),
    re.compile(r"please provide a description of this position\s*:?", re.I),
    re.compile(r"projects?\s*&?\s*deliverables?\s*(may include)?\s*:?", re.I),
    re.compile(r"position overview\s*:?", re.I),
    re.compile(r"responsibilities\s*:?", re.I),
]
NEXT_SECTION_HEADER_RE = re.compile(
    r"(intern will learn|what do you predict|qualifications|work schedule|"
    r"location\s*:|^\s*\d+\.\s|compensation)", re.I | re.M
)
MIN_CHARS_AFTER_ANCHOR_TO_COUNT_AS_CONTENT = 25
ORG_BOILERPLATE_OPENING_RE = re.compile(
    r"^\s*(about\s+[\w\s]+:|who we are\s*:|recently rated|is a multifaith|"
    r"is a nonprofit|is a non-profit)", re.I
)


def extract_duties_text(description: str) -> tuple:
    if pd.isna(description):
        return "", False, True
    text = str(description)

    for pattern in DUTIES_ANCHOR_PATTERNS:
        match = pattern.search(text)
        if not match:
            continue
        after_anchor = text[match.end():]
        next_marker = NEXT_SECTION_HEADER_RE.search(after_anchor)
        extracted = after_anchor[: next_marker.start()] if next_marker else after_anchor
        extracted = extracted.strip()

        if len(extracted) < MIN_CHARS_AFTER_ANCHOR_TO_COUNT_AS_CONTENT:
            return extracted, True, True

        return extracted, True, False

    boilerplate_match = ORG_BOILERPLATE_OPENING_RE.search(text)
    if boilerplate_match:
        first_break = text.find(".", boilerplate_match.end())
        if first_break != -1 and first_break < len(text) - 1:
            text = text[first_break + 1:].strip()

    return text, False, False


def build_occupation_text(role_name: str, role_description: str) -> tuple:
    role_name = "" if pd.isna(role_name) else str(role_name).strip()
    duties_text, anchor_found, blank_intake_form = extract_duties_text(role_description)
    combined = f"{role_name}. {duties_text}"
    return combined[:MAX_DESCRIPTION_CHARS], anchor_found, blank_intake_form


def call_nioccs(industry_text: str, occupation_text: str) -> dict:
    params = {**REQUEST_PARAMS_STATIC, "i": industry_text, "o": occupation_text}
    last_error = None
    for attempt in range(1, MAX_RETRIES_PER_ROW + 1):
        try:
            resp = requests.get(NIOCCS_BASE_URL, params=params, timeout=20)
            if resp.status_code == 200:
                try:
                    return {"raw_response": resp.json(), "http_status": 200, "error": None}
                except json.JSONDecodeError:
                    last_error = f"Non-JSON response (status 200): {resp.text[:200]}"
            else:
                last_error = f"HTTP {resp.status_code}: {resp.text[:200]}"
        except requests.exceptions.RequestException as e:
            last_error = f"Request exception: {e}"
        if attempt < MAX_RETRIES_PER_ROW:
            time.sleep(1.5 * attempt)
    return {"raw_response": None, "http_status": None, "error": last_error}


def parse_nioccs_response(raw) -> dict:
    empty = {
        "naics_code": None, "naics_title": None, "naics_probability": None,
        "soc_code": None, "soc_title": None, "soc_probability": None,
        "naics_insufficient_info": None, "soc_insufficient_info": None,
        "implausible_pairing_flag": None,
    }
    if not isinstance(raw, dict):
        return empty

    industry = raw.get("Industry") or []
    occupation = raw.get("Occupation") or []
    industry_rec = industry[0] if industry else {}
    occupation_rec = occupation[0] if occupation else {}

    naics_code = industry_rec.get("NAICSCode")
    soc_code = occupation_rec.get("SOCCode")

    return {
        "naics_code": naics_code,
        "naics_title": industry_rec.get("NAICSTitle"),
        "naics_probability": industry_rec.get("NAICSProbability"),
        "soc_code": soc_code,
        "soc_title": occupation_rec.get("SOCTitle"),
        "soc_probability": occupation_rec.get("SOCProbability"),
        "naics_insufficient_info": (naics_code == NAICS_INSUFFICIENT_INFO_CODE) if naics_code else None,
        "soc_insufficient_info": (soc_code == SOC_INSUFFICIENT_INFO_CODE) if soc_code else None,
        "implausible_pairing_flag": raw.get("UnexpectedCodeCombination") == "Y",
    }


def load_already_coded():
    if os.path.exists(CHECKPOINT_PATH):
        done = pd.read_csv(CHECKPOINT_PATH)
        genuinely_done = done[
            done["API Call Error"].isna() | (done["Flagged As Blank Intake Form"] == True)
        ]
        return genuinely_done, set(genuinely_done["Opportunity Identifier"])
    return pd.DataFrame(), set()


def run():
    df = pd.read_csv(READY_PATH, encoding="utf-8-sig")

    if TEST_MODE:
        df = df.head(TEST_SAMPLE_SIZE)
        print(f"TEST MODE: running on {len(df)} rows only. Set TEST_MODE = False for the full batch.\n")

    done_df, done_ids = load_already_coded()
    if not TEST_MODE and done_ids:
        print(f"Resuming: {len(done_ids)} rows already coded, skipping those.")
        df = df[~df["Opportunity Identifier"].isin(done_ids)]

    results = []
    total = len(df)
    for i, (_, row) in enumerate(df.iterrows(), start=1):
        opportunity_id = row["Opportunity Identifier"]
        role_name = row["Role Name"]
        role_description = row["Role Description"]
        industry_text = row["Program Hub Category"]  # "Arts, Entertainment, and Recreation" for every row

        occupation_text, duties_anchor_found, blank_intake_form = build_occupation_text(role_name, role_description)

        if blank_intake_form:
            result_row = {
                "Opportunity Identifier": opportunity_id, "Role Name": role_name,
                "Program Hub Category": industry_text,
                "NAICS Code": None, "NAICS Title": None, "NAICS Match Probability": None,
                "SOC Code": None, "SOC Title": None, "SOC Match Probability": None,
                "NAICS Flagged As Insufficient Information": None,
                "SOC Flagged As Insufficient Information": None,
                "Flagged As Implausible NAICS/SOC Pairing": None,
                "Duties Anchor Found In Description": duties_anchor_found,
                "Flagged As Blank Intake Form": True,
                "API Call Error": "Skipped - description is an unfilled intake-form template",
            }
            results.append(result_row)
            if TEST_MODE:
                print(f"--- Row {i}/{total}: {role_name!r} --- SKIPPED (blank intake form)\n")
            continue

        api_result = call_nioccs(industry_text, occupation_text)
        parsed = parse_nioccs_response(api_result["raw_response"])

        result_row = {
            "Opportunity Identifier": opportunity_id, "Role Name": role_name,
            "Program Hub Category": industry_text,
            "NAICS Code": parsed["naics_code"], "NAICS Title": parsed["naics_title"],
            "NAICS Match Probability": parsed["naics_probability"],
            "SOC Code": parsed["soc_code"], "SOC Title": parsed["soc_title"],
            "SOC Match Probability": parsed["soc_probability"],
            "NAICS Flagged As Insufficient Information": parsed["naics_insufficient_info"],
            "SOC Flagged As Insufficient Information": parsed["soc_insufficient_info"],
            "Flagged As Implausible NAICS/SOC Pairing": parsed["implausible_pairing_flag"],
            "Duties Anchor Found In Description": duties_anchor_found,
            "Flagged As Blank Intake Form": False,
            "API Call Error": api_result["error"],
        }
        results.append(result_row)

        if TEST_MODE:
            print(f"--- Row {i}/{total}: {role_name!r} ---")
            print("Occupation text sent:", occupation_text[:200])
            print("Parsed:", {k: v for k, v in result_row.items() if k not in ("Opportunity Identifier", "Role Name")})
            print()
        elif i % 100 == 0:
            print(f"...{i}/{total} rows coded")

        time.sleep(SECONDS_BETWEEN_REQUESTS)

        if not TEST_MODE and i % 100 == 0:
            combined = pd.concat([done_df, pd.DataFrame(results)], ignore_index=True)
            combined.to_csv(CHECKPOINT_PATH, index=False)
            print(f"Checkpoint saved: {len(combined)} rows coded so far.")

    results_df = pd.DataFrame(results)

    if TEST_MODE:
        print(f"\nTest run complete on {len(results_df)} rows. Nothing written to disk yet - "
              f"review the output above, then set TEST_MODE = False and re-run for the full batch.")
        return results_df

    combined = pd.concat([done_df, results_df], ignore_index=True)
    combined.to_csv(CHECKPOINT_PATH, index=False)
    n_errors = combined["API Call Error"].notna().sum()
    print(f"\nDone: {len(combined)} rows written to {CHECKPOINT_PATH}")
    if n_errors:
        print(f"{n_errors} rows had a call error or were skipped as blank intake forms.")

    return combined


results_df = run()